In [1]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [2]:
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
print(torch)
print(torch.__file__)
print(dir(torch)[:40])
print(torch.__version__)


from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import neuroprobe
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

<module 'torch' from 'C:\\ProgramData\\anaconda3\\envs\\neuro\\Lib\\site-packages\\torch\\__init__.py'>
C:\ProgramData\anaconda3\envs\neuro\Lib\site-packages\torch\__init__.py
['AVG', 'AcceleratorError', 'AggregationType', 'AliasDb', 'AnyType', 'Argument', 'ArgumentSpec', 'AwaitType', 'BFloat16Storage', 'BFloat16Tensor', 'BenchmarkConfig', 'BenchmarkExecutionStats', 'Block', 'BoolStorage', 'BoolTensor', 'BoolType', 'BufferDict', 'ByteStorage', 'ByteTensor', 'CallStack', 'Capsule', 'CharStorage', 'CharTensor', 'ClassType', 'Code', 'CompilationUnit', 'CompleteArgumentSpec', 'ComplexDoubleStorage', 'ComplexFloatStorage', 'ComplexType', 'ConcreteModuleType', 'ConcreteModuleTypeBuilder', 'DeepCopyMemoTable', 'DeserializationStorageContext', 'DeviceObjType', 'DictType', 'DisableTorchFunction', 'DisableTorchFunctionSubclass', 'DispatchKey', 'DispatchKeySet']
2.12.1+cpu


In [ ]:
import neuroprobe.config as neuroprobe_config

print("BrainTreebank root:", ROOT_DIR_BRAINTREEBANK)
print("Neuroprobe ROOT_DIR:", neuroprobe_config.ROOT_DIR)
print("Sampling rate:", neuroprobe_config.SAMPLING_RATE, "Hz")

# Subject setup
subject_id = 1
trial_id = 1
coordinates_type = "mni"  # "mni", "mni305", "cortical", "lpi"

subject = BrainTreebankSubject(
    subject_id=subject_id,
    allow_corrupted=False,
    cache=True,
    dtype=torch.float32,
    coordinates_type=coordinates_type,
)
print("Loaded subject", subject_id)
print("First 10 electrode labels:", subject.electrode_labels[:10])
print("First 10 electrode MNI coordinates:")
print(subject.get_electrode_coordinates()[:10])

# Benchmark dataset setup
eval_name = "volume"
output_indices = False
start_neural_data_before_word_onset = 0
end_neural_data_after_word_onset = neuroprobe_config.SAMPLING_RATE * 1  # 1 second

benchmark_dataset = BrainTreebankSubjectTrialBenchmarkDataset(
    subject,
    trial_id,
    dtype=torch.float32,
    eval_name=eval_name,
    output_indices=output_indices,
    start_neural_data_before_word_onset=start_neural_data_before_word_onset,
    end_neural_data_after_word_onset=end_neural_data_after_word_onset,
    lite=True,
)

data_electrode_labels = benchmark_dataset.electrode_labels
data_electrode_coordinates = benchmark_dataset.electrode_coordinates

print("Dataset type:", type(benchmark_dataset))
print("Dataset length:", len(benchmark_dataset))

first_item = benchmark_dataset[0]
print("First item type:", type(first_item))
print("First item:", first_item)

if isinstance(first_item, dict):
    print("First item data shape:", first_item["data"].shape)
    print("First item label:", first_item["label"])
else:
    print("First item shape:", first_item[0].shape)
    print("First item label:", first_item[1])

print("Number of electrodes in dataset:", len(data_electrode_labels))

BrainTreebank root: C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank
Neuroprobe ROOT_DIR: C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank
Sampling rate: 2048 Hz
Loaded subject 1
First 10 electrode labels: ['F3aOFa2', 'F3aOFa3', 'F3aOFa4', 'F3aOFa7', 'F3aOFa8', 'F3aOFa9', 'F3aOFa10', 'F3aOFa11', 'F3aOFa12', 'F3aOFa13']
First 10 electrode MNI coordinates:
tensor([[  8.0828,  44.3820, -15.1744],
        [ 12.4152,  43.6956, -14.8598],
        [ 15.7043,  42.0086, -13.4021],
        [ 27.5911,  38.1577,  -9.0090],
        [ 30.7824,  37.5062,  -7.6428],
        [ 35.1301,  35.9065,  -6.1294],
        [ 38.4192,  34.2195,  -4.6717],
        [ 42.6692,  33.6553,  -3.2497],
        [ 45.9583,  31.9683,  -1.7920],
        [ 50.2907,  31.2818,  -1.4774]])


In [ ]:
import neuroprobe.train_test_splits as neuroprobe_train_test_splits

folds = neuroprobe_train_test_splits.generate_splits_within_session(
    subject,
    trial_id,
    eval_name,
    dtype=torch.float32,
    output_indices=output_indices,
    start_neural_data_before_word_onset=start_neural_data_before_word_onset,
    end_neural_data_after_word_onset=end_neural_data_after_word_onset,
    lite=True,
)

fold_idx = 0
fold = folds[fold_idx]
train_dataset = fold["train_dataset"]
test_dataset = fold["test_dataset"]

def dataset_to_features(ds):
    X_rows = []
    y_rows = []
    for item in ds:
        # item is a dict: {'data': ..., 'label': ..., ...}
        if isinstance(item, dict):
            x = item["data"]
            y = item["label"]
        else:
            x = item[0]
            y = item[1]

        x_np = x.numpy() if torch.is_tensor(x) else np.asarray(x)  # (n_electrodes, n_samples)

        # Summary features per electrode
        mean_per_electrode = x_np.mean(axis=1)   # shape: (n_electrodes,)
        std_per_electrode = x_np.std(axis=1)    # shape: (n_electrodes,)
        max_per_electrode = x_np.max(axis=1)    # shape: (n_electrodes,)

        features = np.concatenate([
            mean_per_electrode,
            std_per_electrode,
            max_per_electrode,
        ])  # shape: (3 * n_electrodes,)

        X_rows.append(features)
        y_rows.append(float(y))

    X = np.stack(X_rows)               # (n_examples, 3 * n_electrodes)
    y = np.array(y_rows, dtype=np.float32)
    return X, y

X_train, y_train = dataset_to_features(train_dataset)
X_test, y_test = dataset_to_features(test_dataset)

print("Compressed X_train shape:", X_train.shape)
print("Compressed X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

alphas = np.logspace(-2, 3, 6)

ridge_cv = Pipeline(
    steps=[
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("ridge", RidgeCV(alphas=alphas)),
    ]
)

ridge_cv.fit(X_train, y_train)
best_alpha = ridge_cv.named_steps["ridge"].alpha_
print("Best alpha:", best_alpha)

y_train_pred = ridge_cv.predict(X_train)
y_test_pred = ridge_cv.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train R^2 (compressed): {train_r2:.3f}")
print(f"Test R^2 (compressed):  {test_r2:.3f}")

It's almost obvious the no linear model will be able to make any significant progress. Let's jump right to a baseline CNN

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class NeuroprobeTorchDataset(Dataset):
    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        if isinstance(item, dict):
            x = item["data"]
            y = item["label"]
        else:
            x = item[0]
            y = item[1]

        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        else:
            x = x.to(torch.float32)

        y = torch.tensor(float(y), dtype=torch.float32)
        return x, y


train_torch_ds = NeuroprobeTorchDataset(train_dataset)
test_torch_ds = NeuroprobeTorchDataset(test_dataset)

train_loader = DataLoader(train_torch_ds, batch_size=16, shuffle=True, num_workers=0)
test_loader = DataLoader(test_torch_ds, batch_size=16, shuffle=False, num_workers=0)

sample_x, sample_y = train_torch_ds[0]
print("Sample x shape:", sample_x.shape)  # (n_electrodes, time)
print("Sample y:", sample_y)

In [ ]:
class EEGCNNRegressor(nn.Module):
    def __init__(self, n_electrodes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(n_electrodes, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x.squeeze(-1)


n_electrodes = sample_x.shape[0]
model = EEGCNNRegressor(n_electrodes=n_electrodes).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(model)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)  # shape: (batch, n_electrodes, time)
        y = y.to(device)

        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_y = []
    all_preds = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        preds = model(x)
        loss = criterion(preds, y)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_preds)
    r2 = r2_score(y_true, y_pred)

    return total_loss / len(loader.dataset), r2, y_true, y_pred

In [ ]:
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_r2, y_true, y_pred = evaluate(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test R^2: {test_r2:.4f}"
    )

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
plt.xlabel("True volume")
plt.ylabel("Predicted volume")
plt.title("CNN regression on Neuroprobe volume")
plt.axline((0, 0), slope=1, color="red", linestyle="--")
plt.tight_layout()
plt.show()